# Rebuild the groove

**Groove id:** …  
**Category:** fidelity / remix / mashup  
**Group:** …

Use `drumkit.py` and the Python standard library. Keep `play(...)` as the **last cell**: it is what we listen to.

In [1]:
from IPython.display import Audio
from drumkit import *

def play(wave, sample_rate=SR):
    return Audio(wave, rate=sample_rate, normalize=True, autoplay=True)

## Tempo and grid

In [2]:
bpm = 130
step_s = 60 / bpm / 4   # one step = one sixteenth

## Patterns

One track per instrument. Replace this example with what you hear.

In [3]:
def square_wave(frequency, amplitude, dur_time, sample_rate):

    dur_samples = round(dur_time * sample_rate)
    
    period_time = 1 / frequency
    period_samples = round(period_time * sample_rate)
    ps_half = round(period_samples / 2)

    repeat = round(dur_samples / period_samples)

    return fit(([-amplitude] * ps_half + [amplitude] * ps_half) * repeat, dur_samples)

def sawtooth_wave(frequency, amp, dur_time, sr = 48000):
    dur_samples = round(dur_time * sr)
    period_samples = round(sr / frequency)

    repeat = round(dur_samples / period_samples)
    return fit([amp * x / period_samples for x in range(period_samples)] * repeat, dur_samples)

def triangle_wave(frequency, amplitude, dur_time, sr = 48000):
    dur_samples = round(dur_time * sr)
    
    period_samples = round(sr / frequency)
    n_half = round(period_samples / 2)

    up = [-1 + 2 * x / n_half for x in range(n_half)]
    down = [1 - 2 * x / n_half for x in range(n_half)]

    # account for unlucky rounding
    if 2 * n_half < period_samples:
        down = down + [down[-1]]

    repeat = round(dur_samples / period_samples)
    
    tri_mult = (up + down) * repeat
    # apply amplitude
    triangle = vca(fit(tri_mult, dur_samples), [amplitude] * dur_samples)

    return triangle

def freq_from_semitones(base_frequency = 440, semitone = 0):
    return base_frequency * (2 ** (semitone / 12))

def render_bass(base_frequency, sequence, steps_time, sample_rate = 48000):

    bass_waves_full = [
        mix(
            triangle_wave(
                    freq_from_semitones(base_frequency, note), 
                    0.2, 
                    step_bass, 
                    sample_rate
                ), 
            sawtooth_wave(
                freq_from_semitones(base_frequency, note),
                0.
            )
        ) for note in bass_sequence
        ]
    
    bass_waves_env = [
        vca(basswave, 
            envelope_exp(step_bass, tau = 0.2, sample_rate = sample_rate)
           ) for basswave in bass_waves_full
    ]
    
    bass_waves_fade = [
        edge_fade(basswave, attack = 0.001, release = 0.0005, sample_rate = sample_rate)
        for basswave in bass_waves_env
    ]
    
    return sum(bass_waves_fade, [])


# define bass sequence
bass_sequence = [
    0, 12, 0, 12, 7, 19, 7, 19, 9, 21, 9, 21, 9, 21, 9, 21, 
    2, 15, 2, 15, 7, 19, 7, 19, 9, 21, 9, 21, 9, 21, 9, 21,
    0, 12, 0, 12, 7, 19, 7, 19, 9, 21, 9, 21, 9, 21, 9, 21,
    2, 15, 2, 15, 0, 12, 0, 12, -1, 11, -1, 11, -1, 11, -1, 11
]
step_bass = step_s * 2

bassline = render_bass(43.65, bass_sequence, step_bass)
# play(bassline)

In [4]:
#  1 e & a 2 e & a 3 e & a 4 e & a
intro_tracks = [
    (("x...x...xxxxxxxx" + "x..." * 4) * 2, kick, step_s)
]

tracks = [
    (("x." * 2) * 4, kick, step_s * 4),
    ((".x" * 2) * 4, snare_mid, step_s * 4),
    (("..x....." + "x..x.x.x." + ".xx...x" + "...xx.x." + "..............x." + ".xx...x...xx.x."), snare, step_s),
    (("xx.x.x.x" * 2) * 4, closed_hat, step_s),
    (("..x...x." * 2) * 4, open_hat, step_s)
]

bridge_tracks = [
    (("xxxxx.x." + "xxxxx.xx" + "xxx.xxxx" + "x.x....."), kick, step_s),
    (("xxxxx.x." + "xxxxx.xx" + "xxx.xxxx" + "x.x....."), snare, step_s),
    (("......xx"), snare_deep, step_s * 4)
]


bar = render_tracks(intro_tracks) + mix(bassline, render_tracks(tracks) * 2, 1, 0.8) + render_tracks(bridge_tracks)

## Listen

In [5]:
play(bar)